# Lesson 01 — Images are arrays

Implement grayscale conversion and histogram equalization, then compare them
with OpenCV.

Read [`README.md`](README.md), then run the setup cell.

In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python notebook kernel is ready.")
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("OpenCV path:", cv2.__file__)

RESULTS_ROOT = Path("results")
RUN_STUDENT_CHECKS = False
print("Student checks enabled:", RUN_STUDENT_CHECKS)

In [ ]:
def make_test_image(height=180, width=260):
    y, x = np.mgrid[0:height, 0:width]
    blue = np.clip(25 + 0.75 * x, 0, 255)
    green = np.clip(20 + 1.10 * y, 0, 255)
    red = np.clip(230 - 0.55 * x + 0.20 * y, 0, 255)
    image = np.dstack([blue, green, red]).astype(np.uint8)
    cv2.rectangle(image, (25, 25), (100, 105), (20, 220, 245), -1)
    cv2.circle(image, (190, 92), 44, (230, 60, 35), -1)
    return image


bgr = make_test_image()
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

print("shape:", bgr.shape)
print("dtype:", bgr.dtype)
print("range:", int(bgr.min()), "to", int(bgr.max()))
print("pixel at row 50, column 70 [B, G, R]:", bgr[50, 70].tolist())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bgr)
axes[0].set_title("Wrong display: BGR labeled as RGB")
axes[1].imshow(rgb)
axes[1].set_title("Correct display conversion")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

### Check 1 — inspect the array

Write down:

- shape, dtype, and range:
- what changed when BGR was converted to RGB:
- the `[B, G, R]` pixel and your predicted grayscale value:

A correct-looking picture can still have the channels reversed.

In [ ]:
a = np.array([250], dtype=np.uint8)
wrapped = a + np.array([20], dtype=np.uint8)
safe = np.clip(a.astype(np.int16) + 20, 0, 255).astype(np.uint8)

print("uint8 arithmetic:", wrapped.tolist())
print("widen, compute, clip, cast:", safe.tolist())

## Overflow test

The previous cell adds beyond the `uint8` range. Write down:

- the wrapped value:
- why it wrapped:
- where an image-brightness function should widen, clip, and cast:

Keep this result. It shows the bug your code must avoid.

In [ ]:
tiny_bgr = np.array(
    [
        [[0, 0, 0], [255, 255, 255]],
        [[0, 0, 255], [0, 255, 0]],
    ],
    dtype=np.uint8,
)

tiny_float = np.rint(
    0.114 * tiny_bgr[..., 0]
    + 0.587 * tiny_bgr[..., 1]
    + 0.299 * tiny_bgr[..., 2]
).astype(np.uint8)
tiny_cv = cv2.cvtColor(tiny_bgr, cv2.COLOR_BGR2GRAY)

print("weighted float result:\n", tiny_float)
print("OpenCV result:\n", tiny_cv)
print("absolute difference:\n", cv2.absdiff(tiny_float, tiny_cv))

## Exercise 1 — grayscale checks

Implement both functions without calling `cv2.cvtColor`.

**Float version:** use the standard BGR weights. Compute in floating point,
round, clip if needed, and return `uint8`. The maximum difference from OpenCV
must be at most 1.

**Integer version:** reproduce OpenCV's fixed-point calculation:

\[
Y = (9798R + 19235G + 3735B + 16384) \; >> \; 15
\]

Widen before multiplication. This version must match OpenCV exactly.

In [ ]:
def student_gray_float(image_bgr):
    """Return uint8 grayscale without cv2.cvtColor."""
    # TODO: implement the weighted floating-point path.
    return None


def student_gray_exact(image_bgr):
    """Return OpenCV-parity uint8 grayscale using the fixed-point rule."""
    # TODO: widen, apply the 15-bit integer coefficients, shift, and cast.
    return None

In [ ]:
reference_gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

if RUN_STUDENT_CHECKS:
    gray_float = student_gray_float(bgr)
    gray_exact = student_gray_exact(bgr)
    assert isinstance(gray_float, np.ndarray)
    assert gray_float.shape == bgr.shape[:2]
    assert gray_float.dtype == np.uint8
    float_max_diff = int(cv2.absdiff(gray_float, reference_gray).max())
    assert float_max_diff <= 1, f"float rung max diff is {float_max_diff}"
    assert np.array_equal(gray_exact, reference_gray), (
        "fixed-point rung is not exact"
    )
    print("PASS — float max difference:", float_max_diff)
    print("PASS — fixed-point difference: 0")
else:
    print("Checks skipped. Implement both functions, then set "
          "RUN_STUDENT_CHECKS = True in the setup cell.")

### Checkpoint 2 — trace one pixel

Choose one pixel where the float result differs from OpenCV, or state that
none differed in this test image.

- coordinate `(row, column)`:
- BGR values:
- unrounded float calculation:
- float result:
- fixed-point result:
- OpenCV result:
- why the float implementation can still be correct:

In [ ]:
def student_equalize(gray):
    """Equalize a uint8 grayscale image using a histogram and LUT."""
    # TODO:
    # 1. Build the 256-bin histogram.
    # 2. Form its cumulative sum.
    # 3. Ignore leading zero-count bins when normalizing.
    # 4. Build a uint8 lookup table and apply it.
    # Match cv2.equalizeHist exactly on the generated image.
    return None

In [ ]:
low_contrast = np.clip(reference_gray // 4 + 85, 0, 255).astype(np.uint8)
cv_equalized = cv2.equalizeHist(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low contrast")
axes[0, 1].imshow(cv_equalized, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("OpenCV equalized reference")
axes[1, 0].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[1, 0].set_title("Before histogram")
axes[1, 1].hist(cv_equalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After histogram")
plt.tight_layout()
plt.show()

if RUN_STUDENT_CHECKS:
    student_eq = student_equalize(low_contrast)
    assert isinstance(student_eq, np.ndarray)
    assert student_eq.dtype == np.uint8
    assert np.array_equal(student_eq, cv_equalized), (
        f"equalization max diff: "
        f"{int(cv2.absdiff(student_eq, cv_equalized).max())}"
    )
    print("PASS — histogram equalization is exact")
else:
    print("Equalization check skipped.")

In [ ]:
if RUN_STUDENT_CHECKS:
    repeats = 25

    start = perf_counter()
    for _ in range(repeats):
        student_gray_exact(bgr)
    student_ms = (perf_counter() - start) * 1000 / repeats

    start = perf_counter()
    for _ in range(repeats):
        cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    opencv_ms = (perf_counter() - start) * 1000 / repeats

    results_dir = RESULTS_ROOT / "lesson-01"
    results_dir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(results_dir / "input.png"), bgr)
    cv2.imwrite(
        str(results_dir / "student-equalized.png"),
        student_equalize(low_contrast),
    )
    (results_dir / "measurements.txt").write_text(
        f"float_max_diff={float_max_diff}\n"
        f"fixed_point_max_diff=0\n"
        f"equalization_max_diff=0\n"
        f"student_grayscale_ms={student_ms:.6f}\n"
        f"opencv_grayscale_ms={opencv_ms:.6f}\n",
        encoding="utf-8",
    )
    print(f"Saved results to {results_dir}")
    print(f"Student: {student_ms:.4f} ms | OpenCV: {opencv_ms:.4f} ms")
else:
    print("Run the checks before saving results.")

## Results and questions

Files:

- `results/lesson-01/input.png`
- `results/lesson-01/student-equalized.png`
- `results/lesson-01/measurements.txt`

Write down:

- float maximum difference:
- integer maximum difference:
- equalization maximum difference:
- student/OpenCV runtime and repeat count:
- one visible effect of equalization:

Answer:

1. Why are the grayscale weights not one third each?
2. Why can the float version differ by 1?
3. What causes `uint8` wraparound?
4. Trace one intensity through your equalization lookup table.
5. When could global equalization make an image worse?

## Fixes

| Problem | Check | Next step |
| --- | --- | --- |
| Colors look swapped | Channel order | Convert BGR to RGB only for display |
| Bright values become dark | Array dtype | Widen, compute, clip, cast |
| Float difference exceeds 1 | One failing BGR pixel | Check channel order and rounding |
| Integer difference is nonzero | Intermediate dtype | Use a wide integer before multiplication |
| Equalization differs | Nonzero histogram bins | Handle the first nonzero CDF value |

Done when all checks pass with `RUN_STUDENT_CHECKS = True`, the three result
files exist, and the questions are answered from your run.

### Primary sources

- OpenCV, Basic Operations on Images:
  <https://docs.opencv.org/master/d3/df2/tutorial_py_basic_ops.html>
- OpenCV, Histogram Equalization:
  <https://docs.opencv.org/master/d4/d1b/tutorial_histogram_equalization.html>
- OpenCV, Histogram Calculation:
  <https://docs.opencv.org/master/d8/dbc/tutorial_histogram_calculation.html>
- OpenCV source, grayscale conversion constants:
  <https://github.com/opencv/opencv/blob/4.x/modules/imgproc/src/color.simd_helpers.hpp>
- NumPy quickstart:
  <https://numpy.org/doc/stable/user/quickstart.html>
- NumPy `bincount`:
  <https://numpy.org/doc/stable/reference/generated/numpy.bincount.html>

Accessed 2026-07-25.